# Loading Spectra with pySpectrum

pySpectrum handles 1D detector spectra from two common data formats:

| Format | Description | Loader |
|---|---|---|
| **Channel–counts** | Pre-histogrammed CSV or DataFrame | `Spectrum.from_dataframe` |
| **Time–channel list-mode** | Raw event-by-event DAQ output | `TimeChannelParser` |

Both produce a calibrated `Spectrum` backed by xarray — Poisson errors, axis calibration, and resolution calibration are attached at load time and propagated automatically through all downstream operations.

## Workflow
1. Set up energy and resolution calibration
2. Load a channel–counts CSV → `Spectrum`
3. Load a time–channel list-mode file → `Spectrum` (in-memory or chunked from disk)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scispectrum.core import Spectrum
from scispectrum.calibration import AxisCalibration, ResolutionCalibration
from scispectrum.calibration.models.hpge_fwhm_model import StandardHPGeFWHMModel
from scispectrum.io import TimeChannelParser

## 1. Set up calibration

`AxisCalibration` maps raw channel numbers to physical axis values (energy in keV).  
`ResolutionCalibration` returns the detector FWHM at any axis value — used by peak-finding and fitting routines to set analysis window sizes.

> **Adapt to your detector:** replace the polynomial coefficients and FWHM value below. These are typically determined by fitting known γ-ray lines from a calibration source (e.g. ⁶⁰Co, ¹³³Ba, ¹⁵²Eu). See the *Calibration* notebook for an automated approach.

In [ ]:
# ── Adapt to your detector ────────────────────────────────────────────────────
energy_calib_poly      = np.poly1d([0.0408976444, 0.0822321508])  # channel → keV
energy_resolution_fwhm = 1.08   # FWHM at 511 keV [keV]
# ─────────────────────────────────────────────────────────────────────────────

energy_calib = AxisCalibration(func=energy_calib_poly, name="energy")

# FWHM scales as sqrt(E / 511 keV) — standard approximation for HPGe detectors
estimated_FWHM = StandardHPGeFWHMModel().generator((0, energy_resolution_fwhm / 511**0.5, 0))
res_calib = ResolutionCalibration(func=estimated_FWHM)

## 2. Channel–counts CSV

The simplest format: a table of channel indices and measured counts. `Spectrum.from_dataframe` accepts any pandas DataFrame. Providing a `counts_error` column enables Poisson uncertainty propagation through all downstream operations.

> **Adapt:** adjust `path_csv` and the `read_csv` parameters (separator, column names, header rows) to match your file format.

In [ ]:
# ── Adapt to your file ────────────────────────────────────────────────────────
path_csv = '../Library/152Eu_calsource_10cm_85ks.txt'
# ─────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(path_csv, names=['counts'])
df['channel']       = df.index
df['counts_error']  = df['counts'] ** 0.5   # Poisson errors

spectrum_csv = Spectrum.from_dataframe(
    df=df,
    channel_col='channel',
    counts_col='counts',
    counts_error_col='counts_error',
    axis_calib=energy_calib,
    resolution_calib=res_calib,
)
print(f"Loaded {len(spectrum_csv.counts):,} channels, "
      f"axis range: {spectrum_csv.axis[0]:.1f} – {spectrum_csv.axis[-1]:.1f} keV")

In [ ]:
spectrum_csv.data.plot(yscale='log')
plt.title('¹⁵²Eu spectrum — loaded from channel–counts CSV')
plt.xlabel('Energy [keV]')
plt.ylabel('Counts')
plt.grid(True, which='both')
plt.tight_layout()
plt.show()

## 3. Time–channel list-mode

List-mode data stores individual detector events — each row is one detected particle (time stamp, channel, optional pile-up flag). `TimeChannelParser` histograms these events into a `Spectrum`, automatically filtering:
- channels ≤ 0 (noise / underflow)
- events flagged as pile-up or overflow (`flag ≠ 0`)

Two methods are available:

| Method | When to use |
|---|---|
| `from_dataframe` | Data already loaded in memory |
| `from_file` | Large files — reads in chunks, avoiding memory exhaustion |

### 3a. From a DataFrame

> **Adapt:** adjust `path_listmode`, the `read_csv` parameters, and `num_of_channels` to match your DAQ format and detector channel count.

In [ ]:
# ── Adapt to your file ────────────────────────────────────────────────────────
path_listmode   = '../Library/time_channel_to_spectrum.txt'
num_of_channels = 16384   # total detector channels
# ─────────────────────────────────────────────────────────────────────────────

df_listmode = pd.read_csv(path_listmode, sep=' ', skiprows=5,
                           names=['time', 'channel', 'flag'], usecols=[0, 1, 2])

spectrum_df = TimeChannelParser.from_dataframe(
    df_listmode,
    axis_calib=energy_calib,
    resolution_calib=res_calib,
    num_of_channels=num_of_channels,
)
print(f"Events in file: {len(df_listmode):,} → valid counts: {spectrum_df.counts.sum():,.0f}")

### 3b. From a file (chunked — recommended for large datasets)

`from_file` streams the data in `chunk_size` rows at a time, accumulating the histogram without loading the full file into RAM. The resulting `Spectrum` is identical to `from_dataframe`.

> **Adapt:** set `chunk_size` based on available memory. Values of 10 000 – 1 000 000 are typical.

In [ ]:
# ── Adapt chunk size to available memory ──────────────────────────────────────
chunk_size = 10_000
# ─────────────────────────────────────────────────────────────────────────────

spectrum_file = TimeChannelParser.from_file(
    path_listmode,
    axis_calib=energy_calib,
    resolution_calib=res_calib,
    num_of_channels=num_of_channels,
    chunk_size=chunk_size,
    sep=' ', skiprows=5, names=['time', 'channel', 'flag'], usecols=[0, 1, 2],
)

In [ ]:
# Both methods produce identical spectra — curves overlap perfectly
spectrum_df.data.plot(yscale='log', label='from_dataframe')
spectrum_file.data.plot(yscale='log', label='from_file (chunked)', ls='--')

plt.xlim([300, 650])
plt.title('Time–channel spectrum — positron annihilation (~511 keV)')
plt.xlabel('Energy [keV]')
plt.ylabel('Counts')
plt.grid(True, which='both')
plt.legend()
plt.tight_layout()
plt.show()